# 01 · Climate-change signal in significant wave height

**Objective.** Quantify how the wave climate simulated by WAVEWATCH III / EC-Earth3 changes at the end of the century (2071–2100) under a low-emission (SSP1-2.6) and a high-emission (SSP5-8.5) pathway, relative to the model's historical climate (1985–2014).

**Questions.**
1. *Where and by how much* do the mean and the high percentiles (p90, p99) of Hs change?
2. *Are the changes larger than natural variability*, i.e. would they stand out from year-to-year fluctuations?
3. *How much does the emission pathway matter*: how different are SSP5-8.5 and SSP1-2.6?

**Principle.** The change at each sea point is Δ = statistic(scenario, 2071–2100) − statistic(historical, 1985–2014). Both periods come from the same model, so most of the model's own bias cancels in the difference (the "delta" approach). Thirty years average out most weather and year-to-year variability. Following Meucci et al. (2024), a change in the mean is called **significant** where |Δ| exceeds the standard deviation of the 30 annual means of the future period (its interannual variability); significant areas are hatched. Regional numbers are area-weighted (cos latitude). Points with more than 10 % missing values (sea ice) in either period are masked, because there the statistics describe only the ice-free time.

**Inputs → outputs.** `data/processed/stats30/*.npz` from `src/compute_stats.py` → figures in `fig/change/`, tables in `data/processed/change/`.

**Answer.** *(written last, see section 7)*

## 0. Setup
Run first. `VAR` selects the variable: `hs` (default), `wspd` (wind speed) or `tp` (peak period), once `compute_stats.py` has produced it for all three scenarios.

In [ ]:
%reload_ext autoreload
%autoreload 2
import sys
import warnings
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML
from cmip_utils import load_grid, plot_map, to_map, area_mean, save_fig, save_table
from compute_stats import load_stats, SCENARIOS

warnings.filterwarnings("ignore", category=RuntimeWarning)

def table(df, digits=3):
    """Show a DataFrame as a plain HTML table (works with any VS Code table viewer)."""
    display(HTML(df.round(digits).to_html()))       # all-NaN (sea-ice) points
pd.set_option("display.float_format", "{:,.3f}".format)
plt.rcParams.update({"hatch.color": "0.15", "hatch.linewidth": 0.4})

VAR, UNIT, NAME = "hs", "m", "Hs"          # or "wspd", "m/s", "wind speed";  "tp", "s", "Tp"
SSPS = ("ssp126", "ssp585")
LABEL = {"historical": "historical", "ssp126": "SSP1-2.6", "ssp585": "SSP5-8.5"}
MASK_LIMIT = 0.10                          # mask points missing > 10 % of the time
SUB = "change"

grid = load_grid(verbose=False)
st = {s: load_stats(s, VAR, "best") for s in SCENARIOS}
PERIOD = {s: st[s]["period"] for s in SCENARIOS}
print({LABEL[s]: PERIOD[s] for s in SCENARIOS})
if any(PERIOD[s] != "2071-2100" for s in SSPS):
    print("NOTE: SSP statistics cover", {LABEL[s]: PERIOD[s] for s in SSPS},
          "- the damaged 2099-2100 stretch is not repaired yet. Rerun after the repair.")

### Helpers

`show` draws one map with land and masked points in light grey; `hatch` marks significant points. `ok[s]` is True where both the historical period and scenario `s` have at most 10 % missing values.

In [ ]:
def show(values, ax, title, label, **kw):
    plot_map(grid, values, ax=ax, title=title, label=label, **kw)
    ax.set_facecolor("lightgrey")                     # land and masked points

def hatch(ax, mask):
    """Hatch the points where mask is True (significant change)."""
    if np.any(mask):
        ax.contourf(grid["lon"], grid["lat"], to_map(grid, mask.astype(float)),
                    levels=[0.5, 1.5], colors="none", hatches=["...."])

def masked(values, s):
    return np.where(ok[s], values, np.nan)

ok = {s: (st["historical"]["nan_frac"] <= MASK_LIMIT) & (st[s]["nan_frac"] <= MASK_LIMIT) for s in SSPS}
for s in SSPS:
    print(f"{LABEL[s]}: {100 * area_mean(grid, (~ok[s]).astype(float)):.1f} % of the ocean area masked")

## 1. The baseline: historical wave climate (1985–2014)

Before looking at changes, the reference itself: where waves are large on average and in storms. The changes in the next sections are easier to judge against these magnitudes.

In [ ]:
h = st["historical"]
fig, axs = plt.subplots(1, 3, figsize=(20, 4.2), constrained_layout=True)
for ax, key, what in zip(axs, ("mean", "p90", "p99"), ("mean", "90th percentile", "99th percentile")):
    show(h[key], ax, f"historical {PERIOD['historical']}: {what} {NAME}", f"{NAME} [{UNIT}]",
         cmap="viridis", vmin=0, vmax=np.nanpercentile(h[key], 99.5))
save_fig(fig, f"{VAR}_historical_climate", SUB);
plt.show()

## 2. Where and by how much does it change? (Question 1)

Absolute change Δ = scenario − historical, for the mean and the two high percentiles. Both scenarios share one colour scale per column, so their strengths can be compared directly. Hatching (mean only) marks changes larger than the interannual variability.

In [ ]:
STATS = (("mean", "mean"), ("p90", "90th percentile"), ("p99", "99th percentile"))
delta = {s: {k: masked(st[s][k] - st["historical"][k], s) for k, _ in STATS} for s in SSPS}
sigma = {s: np.nanstd(st[s]["annual_mean"], axis=0) for s in SSPS}          # interannual variability
signif = {s: np.abs(delta[s]["mean"]) > sigma[s] for s in SSPS}

fig, axs = plt.subplots(2, 3, figsize=(20, 8), constrained_layout=True)
for j, (key, what) in enumerate(STATS):
    lim = np.nanpercentile(np.abs(np.concatenate([delta[s][key] for s in SSPS])), 99)
    for i, s in enumerate(SSPS):
        show(delta[s][key], axs[i, j], f"{LABEL[s]} − historical: Δ {what}", f"Δ {NAME} [{UNIT}]",
             diverging=True, vmax=lim)
        if key == "mean":
            hatch(axs[i, j], signif[s])
fig.suptitle(f"Change in {NAME}, {PERIOD['ssp585']} vs {PERIOD['historical']} "
             f"(hatched: change larger than interannual variability; grey: land or sea ice)")
save_fig(fig, f"{VAR}_change_maps", SUB);
plt.show()

### Relative change of the mean

The same change as a percentage of the historical mean, as in Meucci et al. (2024, Fig. 6). A 0.3 m change matters more where waves are 1 m than where they are 4 m.

In [ ]:
rel = {s: 100 * delta[s]["mean"] / st["historical"]["mean"] for s in SSPS}
lim = np.nanpercentile(np.abs(np.concatenate(list(rel.values()))), 99)
fig, axs = plt.subplots(1, 2, figsize=(16, 4.2), constrained_layout=True)
for ax, s in zip(axs, SSPS):
    show(rel[s], ax, f"{LABEL[s]}: relative change in mean {NAME}", "Δ [%]", diverging=True, vmax=lim)
    hatch(ax, signif[s])
save_fig(fig, f"{VAR}_change_mean_relative", SUB);
plt.show()

### Change by latitude

Averaging the change along each latitude band summarises the maps in one line per scenario: it shows at a glance which latitudes gain and which lose.

In [ ]:
bands = np.arange(-80, 91, 2)
lat = grid["point_lat"]
fig, ax = plt.subplots(figsize=(10, 4.5), constrained_layout=True)
for s, colour in zip(SSPS, ("tab:blue", "tab:red")):
    d = delta[s]["mean"]
    zonal = [np.nanmean(d[(lat >= a) & (lat < a + 2)]) if ((lat >= a) & (lat < a + 2)).any() else np.nan
             for a in bands[:-1]]
    ax.plot(bands[:-1] + 1, zonal, color=colour, label=LABEL[s])
ax.axhline(0, color="k", lw=0.6)
ax.set(xlabel="latitude [°N]", ylabel=f"Δ mean {NAME} [{UNIT}]",
       title=f"Zonal-mean change in mean {NAME} (masked points excluded)")
ax.legend()
save_fig(fig, f"{VAR}_change_zonal", SUB);
plt.show()

## 3. Is the change larger than natural variability? (Question 2)

Share of the (unmasked) ocean area where the change in the mean exceeds the interannual variability, split into increases and decreases.

In [ ]:
rows = {}
for s in SSPS:
    d, sig = delta[s]["mean"], signif[s] & ok[s]
    rows[LABEL[s]] = {"significant increase [% area]": 100 * area_mean(grid, np.where(ok[s], sig & (d > 0), np.nan)),
                      "significant decrease [% area]": 100 * area_mean(grid, np.where(ok[s], sig & (d < 0), np.nan)),
                      "not significant [% area]": 100 * area_mean(grid, np.where(ok[s], ~sig, np.nan))}
signif_table = pd.DataFrame(rows).T
table(signif_table, 1)
save_table(signif_table, f"{VAR}_significant_area", SUB);

## 4. How much does the emission pathway matter? (Question 3)

Difference between the two futures, SSP5-8.5 − SSP1-2.6, directly (no historical needed). Where it is close to zero, the outcome hardly depends on emissions; where it is large, mitigation makes a difference.

In [ ]:
both = ok["ssp126"] & ok["ssp585"]
fig, axs = plt.subplots(1, 2, figsize=(16, 4.2), constrained_layout=True)
for ax, key, what in zip(axs, ("mean", "p99"), ("mean", "99th percentile")):
    d = np.where(both, st["ssp585"][key] - st["ssp126"][key], np.nan)
    show(d, ax, f"SSP5-8.5 − SSP1-2.6: {what} {NAME}", f"Δ {NAME} [{UNIT}]",
         diverging=True, vmax=np.nanpercentile(np.abs(d), 99))
save_fig(fig, f"{VAR}_pathway_difference", SUB);
plt.show()

## 5. Regional summary

Area-weighted averages over a few ocean regions. The regions are simple latitude–longitude boxes (longitude 0–360 °E); edit `REGIONS` to agree a regional focus with the supervisors.

In [ ]:
lat, lon = grid["point_lat"], grid["point_lon"]
REGIONS = {
    "Global": np.ones_like(lat, bool),
    "Southern Ocean (S of 45°S)": lat < -45,
    "Tropics (15°S–15°N)": np.abs(lat) <= 15,
    "North Pacific (20–60°N, 120°E–110°W)": (lat >= 20) & (lat <= 60) & (lon >= 120) & (lon <= 250),
    "North Atlantic (20–65°N, 80°W–0°)": (lat >= 20) & (lat <= 65) & (lon >= 280),
    "North Sea & Norwegian Sea (50–72°N, 10°W–20°E)": (lat >= 50) & (lat <= 72) & ((lon >= 350) | (lon <= 20)),
    "Arctic (N of 66°N)": lat > 66,
}
rows = []
for s in SSPS:
    for region, sel in REGIONS.items():
        use = sel & ok[s]
        if not use.any():
            continue
        r = {"scenario": LABEL[s], "region": region}
        for key in ("mean", "p99"):
            hist = area_mean(grid, np.where(use, st["historical"][key], np.nan))
            d = area_mean(grid, np.where(use, delta[s][key], np.nan))
            r[f"hist {key} [{UNIT}]"], r[f"Δ {key} [{UNIT}]"], r[f"Δ {key} [%]"] = hist, d, 100 * d / hist
        r["significant [% area]"] = 100 * area_mean(grid, np.where(use, signif[s], np.nan))
        r["masked [% area]"] = 100 * area_mean(grid, np.where(sel, ~ok[s], np.nan))
        rows.append(r)
regional = pd.DataFrame(rows).set_index(["scenario", "region"])
table(regional, 2)
save_table(regional, f"{VAR}_regional_summary", SUB);

## 6. Limitations

- **One model, one ensemble member.** The changes are EC-Earth3's; other CMIP6 models differ (Meucci et al., 2024, show the eight-model spread). Natural variability is sampled by 30 years, not by several runs.
- **Reference.** The baseline is the model's own historical run, not ERA5 (decision by the data provider). It is calibrated (wind factor 1.08) to match the ERA5-based wave climate statistically, but it does not reproduce the actual weather of 1985–2014.
- **Sea ice.** Points missing more than 10 % of the time are masked; elsewhere in polar seas, statistics cover only ice-free time, so retreating ice can raise them by itself.
- **Significance** is tested for the mean only; for p90/p99 the maps show the size of the change, not its significance.
- **Period.** If the SSP period printed in section 0 is 2071–2098, the damaged 2099–2100 stretch is excluded until it is repaired; 28 instead of 30 years has little effect on 30-year statistics.

## 7. Answer

*(Fill in after running; 2–3 sentences per question, citing the figures and the tables.)*

1. **Where and how much:** e.g. "Under SSP5-8.5 the mean Hs increases by up to … m (…%) in the Southern Ocean and decreases by … m in the North Pacific and North Atlantic; the 99th percentile changes by …"
2. **Larger than natural variability:** e.g. "…% of the ocean area shows a significant increase and …% a significant decrease under SSP5-8.5, against …% and …% under SSP1-2.6."
3. **Role of the pathway:** e.g. "The pattern is the same in both scenarios but about … times stronger under SSP5-8.5; the largest differences are in …"
